## Prerequisites & Runtime

**Estimated wall time on CPU**: ~10–15 minutes

**Required**:
- AmorphGen (`pip install amorphgen` or `pip install -e ../..`)
- MACE backend (`pip install mace-torch`) — first run downloads the MPA-0-medium model (~600 MB)
- `matplotlib` for the analysis plots

**System under study**: amorphous In₂O₃ (In₁₆O₂₄, 40 atoms)

**What this case study demonstrates**:
1. Random placement with deliberately-loose O–O minimum separation produces structures containing peroxide-like O–O dimers (~1.47 Å), an unphysical defect for amorphous In₂O₃.
2. High-temperature equilibration dissociates these dimers within the simulation window.
3. The dissociation timescale follows an Arrhenius temperature dependence, validating that the AmorphGen melt-quench protocol successfully heals placement-induced defects.

Position in the tutorial sequence: this is an **application case study** — it assumes familiarity with `generate_random()`, `opt_run`, and `eq_run` covered in tutorials 1–3.

---


In [ ]:
# Install AmorphGen if needed (uncomment one):
#   !pip install amorphgen           # from PyPI (when published)
#   !pip install -e ../..            # editable install from local repo
#
# Plus the MACE backend (the model download is automatic on first use):
#   !pip install mace-torch matplotlib

# Case Study: O–O Dimer Dissociation Kinetics in Amorphous In₂O₃

## Hybrid Workflow: Random Generation → Optimisation → High-Temperature Equilibration

**Objective:** Investigate whether oxygen dimer defects (O–O peroxide bonds, ~1.47 Å) in
amorphous In₂O₃ structures can be dissociated by high-temperature equilibration, and
determine the temperature dependence of the dissociation timescale.

**Background:**

From the previous minsep case study, we found that:
- Generating random In₂O₃ structures with a **loose O–O minsep (1.5 Å)** permits
  O–O contacts near the peroxide bond length (~1.47 Å)
- After optimisation with **MACE-MPA-0-medium**, these dimers persist as metastable
  defects (~0.3 eV/atom higher energy than dimer-free structures)
- **CHGNet** does not support dimers (pushes all O–O > 2.46 Å during optimisation)

**Approach:**

We use MACE-optimised loose-minsep structures containing O–O dimers as starting points,
then run NVT equilibration at multiple temperatures to observe dimer dissociation.

**Arrhenius estimate** (assuming barrier $E_a \approx 1$ eV, attempt frequency $\nu \approx 10^{13}$ Hz):

$$\tau \approx \nu^{-1} \exp\left(\frac{E_a}{k_B T}\right)$$

| Temperature (K) | $k_B T$ (eV) | Estimated $\tau$ |
|:---:|:---:|:---:|
| 1000 | 0.086 | ~11.0 ns |
| 1500 | 0.129 | ~229 ps |
| 2000 | 0.172 | ~33.1 ps |
| 2500 | 0.215 | ~10.4 ps |
| 3000 | 0.259 | ~4.8 ps |
| 3500 | 0.302 | ~2.8 ps |
| 4000 | 0.345 | ~1.8 ps |

At 3000 K and above, dimer dissociation should occur within a few ps — well
within our 4 ps simulation window. We run three temperatures to see the progression.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from pathlib import Path
from ase.io import read, write
from ase.geometry import get_distances

# AmorphGen imports
from amorphgen.pipeline.random_gen import generate_random
from amorphgen.pipeline.opt_cell import run as opt_run
from amorphgen.pipeline.equilibrate import run as eq_run
from amorphgen.utils.calculators import get_calculator

# Plotting defaults
plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 9,
    "figure.figsize": (8, 5),
})

print("Imports complete.")

Imports complete.


## 1. Parameters

We define the system, generation parameters, and the temperature schedule for equilibration.

In [2]:
# --- System definition ---
composition = {"In": 16, "O": 24}   # In₁₆O₂₄ = In₂O₃ (40 atoms)
target_density = 5.5                  # g/cm³ (23% below crystalline 7.12 g/cm³)

# --- Minsep: loose O-O to permit dimers ---
minsep_loose = {"In-In": 2.8, "In-O": 1.9, "O-O": 1.5}

# --- Random generation ---
n_structures = 2
seed = 42
max_attempts_per_atom = 50000

# --- MLIP backend ---
model_name = "mace-mpa-0-medium"
device = "cpu"                        # MPS may hang for MD; use CPU
default_dtype = "float64"             # Required for geometry optimisation with MACE

# --- Optimisation ---
opt_fmax = 0.05                       # eV/Å
opt_max_steps = 500
opt_cell_filter = "cubic"             # Isotropic volume relaxation, shape fixed

# --- Equilibration temperature schedule ---
temperatures = [3000, 4000]  # K — at least 2 for kinetics comparison
eq_timestep = 2.0                     # fs
eq_time_ps = 2.0                      # ps per temperature
eq_steps = int(eq_time_ps * 1000 / eq_timestep)  # 10,000 steps

# --- Dimer detection ---
dimer_cutoff = 2.0  # Å — O-O pairs closer than this are counted as dimers
                     # Peroxide bond ~1.47 Å; typical amorphous O-O > 2.5 Å

# --- Directories ---
base_dir = Path("case_study_dimer_dissociation")
gen_dir = base_dir / "random_loose"
opt_dir = base_dir / "opt_MACE"

for d in [gen_dir, opt_dir]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Equilibration: {len(temperatures)} temperatures × {eq_time_ps} ps = {len(temperatures) * eq_time_ps} ps total")
print(f"Steps per temperature: {eq_steps}")

Equilibration: 1 temperatures × 4.0 ps = 4.0 ps total
Steps per temperature: 2000


## 2. Generate Random Structures (Loose O–O minsep)

Generate structures with O–O minsep = 1.5 Å to permit dimer-like contacts.

In [3]:
structures_random = []

for i in range(n_structures):
    atoms = generate_random(
        composition=composition,
        target_density=target_density,
        minsep=minsep_loose,
        seed=seed + i,
        max_attempts_per_atom=max_attempts_per_atom,
    )
    out_path = gen_dir / f"random_loose_{i+1:02d}.vasp"
    write(str(out_path), atoms, format="vasp")
    structures_random.append(atoms)
    print(f"  Structure {i+1}: {len(atoms)} atoms, volume = {atoms.get_volume():.1f} ų")

print(f"\nGenerated {len(structures_random)} random structures in {gen_dir}/")

  Structure 1: 40 atoms, volume = 670.6 ų

Generated 1 random structures in case_study_dimer_dissociation/random_loose/


## 3. Optimise with MACE (Retains Dimers)

From the previous case study, MACE preserves O–O dimers during optimisation while CHGNet
does not. We use MACE here to obtain dimer-containing optimised structures.

In [4]:
calc = get_calculator(model_name, device=device, default_dtype=default_dtype)

structures_opt = []

for i, atoms in enumerate(structures_random):
    print(f"Optimising structure {i+1}/{n_structures}...")
    opt_atoms = opt_run(
        atoms.copy(),
        calc=calc,
        cfg_override={"opt": {
            "fmax": opt_fmax,
            "max_steps": opt_max_steps,
            "cell_filter": opt_cell_filter,
        }},
    )
    out_path = opt_dir / f"opt_loose_{i+1:02d}.vasp"
    write(str(out_path), opt_atoms, format="vasp")
    structures_opt.append(opt_atoms)

print(f"\nOptimised {len(structures_opt)} structures in {opt_dir}/")

/Users/c.kaewmeechai@bham.ac.uk/Library/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.
[MACE] Loading foundation model 'mace-mpa-0-medium' → mace_mp(model='medium-mpa-0')
Using medium MPA-0 model as default MACE-MP model, to use previous (before 3.10) default model please specify 'medium' as model argument
Using Materials Project MACE for MACECalculator with /Users/c.kaewmeechai@bham.ac.uk/.cache/mace/macempa0mediummodel
Using float64 for MACECalculator, which is slower but more accurate. Recommended for geometry optimization.
Optimising structure 1/1...
[Opt] Using provided Atoms object

  Composition: In16O24 (40 atoms)
  Initial cell: a=8.7529  b=8.7529  c=8.7529
  Volume: 670.59 A^3
  Optimizer: LBFGS  fmax=0.05  max_steps=500
  Cell filter: cubic
  [cell] Cubic: isotropic volume only, shape fixed

   Step      Energy(eV)   Fmax(eV/A)        a(A)        b(A)        c(A)     Vol(A3)
  -----------------------------------------------------------------------------------

/Users/c.kaewmeechai@bham.ac.uk/Library/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/mace/calculators/mace.py:199: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


      1     -136.664063    59.132770   10.503494   10.503494   10.503494   1158.7810
      2     -156.659796    22.171372    9.151903    9.151903    9.151903    766.5389
      3     -149.117174   106.256437    8.166538    8.166538    8.166538    544.6456
      4     -160.400862     6.069078    8.913266    8.913266    8.913266    708.1260
      5     -162.636862     3.706348    8.859529    8.859529    8.859529    695.3956
      6     -166.997964     7.404491    8.768954    8.768954    8.768954    674.2849
      7     -170.010927    19.650036    8.719708    8.719708    8.719708    662.9882
      8     -172.854715    21.170838    8.780446    8.780446    8.780446    676.9393
      9     -175.551324     4.050025    8.964456    8.964456    8.964456    720.3968
     10     -176.953690     7.201787    9.048076    9.048076    9.048076    740.7451
     11     -179.441948    11.383132    9.135734    9.135734    9.135734    762.4834
     12     -180.982457     7.594663    9.171691    9.171691    9

## 4. Dimer Detection

Define a helper function to count O–O dimers (pairs with distance < cutoff) in a structure.
We use ASE's `get_distances` with `mic=True` for correct periodic boundary treatment.

In [5]:
def count_oo_dimers(atoms, cutoff=2.0):
    """
    Count O-O pairs with distance < cutoff.

    Returns
    -------
    n_dimers : int
        Number of O-O pairs below cutoff.
    min_oo : float
        Minimum O-O distance in the structure.
    oo_distances : list of float
        All O-O distances below cutoff.
    """
    o_indices = [i for i, s in enumerate(atoms.get_chemical_symbols()) if s == "O"]
    if len(o_indices) < 2:
        return 0, np.inf, []

    o_positions = atoms.positions[o_indices]

    # Get all O-O distances with minimum image convention
    _, D = get_distances(o_positions, cell=atoms.cell, pbc=True)
    
    n = len(o_indices)
    oo_dists = []
    for ii in range(n):
        for jj in range(ii + 1, n):
            d = np.linalg.norm(D[ii, jj])
            if d < cutoff:
                oo_dists.append(d)

    min_oo = np.min([np.linalg.norm(D[ii, jj]) for ii in range(n) for jj in range(ii+1, n)])

    return len(oo_dists), min_oo, oo_dists


# Verify dimers exist in optimised structures
print("Dimer count in MACE-optimised loose-minsep structures:")
print(f"{'Structure':>12s}  {'Dimers':>6s}  {'Min O-O (Å)':>12s}")
print("-" * 36)

dimer_structures = []
for i, atoms in enumerate(structures_opt):
    n_dim, min_oo, _ = count_oo_dimers(atoms, cutoff=dimer_cutoff)
    print(f"  {i+1:>8d}  {n_dim:>6d}  {min_oo:>12.3f}")
    if n_dim > 0:
        dimer_structures.append((i, atoms.copy()))

print(f"\n{len(dimer_structures)} / {len(structures_opt)} structures contain dimers.")
if len(dimer_structures) == 0:
    print("WARNING: No dimers found! Check minsep or try different seeds.")

Dimer count in MACE-optimised loose-minsep structures:
   Structure  Dimers   Min O-O (Å)
------------------------------------
         1       5         1.467

1 / 1 structures contain dimers.


## 5. High-Temperature Equilibration

Run NVT equilibration on each dimer-containing structure at multiple temperatures.
We use NVT to preserve the target density (NPT would allow volume changes).

For each temperature, we track the O–O dimer count over the trajectory to observe
dissociation kinetics.

**Note:** This is the most computationally expensive step. On CPU with MACE-MPA-0-medium,
expect ~30 sec to 1 min per structure per temperature (4 ps each). Total: ~5–15 min for
5 structures × 3 temperatures.

In [6]:
def track_dimers_in_trajectory(traj_path, cutoff=2.0, stride=10):
    """
    Read a trajectory file and count O-O dimers at each frame.
    
    Parameters
    ----------
    traj_path : str or Path
        Path to trajectory file (extxyz/xyz format).
    cutoff : float
        O-O distance cutoff for dimer detection (Å).
    stride : int
        Read every Nth frame to speed up analysis.
    
    Returns
    -------
    times_ps : np.ndarray
        Time in ps for each analysed frame.
    n_dimers : np.ndarray
        Number of O-O dimers at each frame.
    min_oo : np.ndarray
        Minimum O-O distance at each frame.
    """
    traj = read(str(traj_path), index=f"::{stride}")
    
    times = []
    dimers = []
    min_oos = []
    
    for frame_idx, atoms in enumerate(traj):
        # Time from trajectory step count
        t_ps = frame_idx * stride * eq_timestep / 1000.0  # fs -> ps
        n_dim, min_oo, _ = count_oo_dimers(atoms, cutoff=cutoff)
        times.append(t_ps)
        dimers.append(n_dim)
        min_oos.append(min_oo)
    
    return np.array(times), np.array(dimers), np.array(min_oos)


print(f"Will equilibrate {len(dimer_structures)} dimer-containing structures")
print(f"at temperatures: {temperatures} K")
print(f"Duration: {eq_time_ps} ps per run ({eq_steps} steps, dt = {eq_timestep} fs)")

Will equilibrate 1 dimer-containing structures
at temperatures: [4000] K
Duration: 4.0 ps per run (2000 steps, dt = 2.0 fs)


In [7]:
# Storage for results: results[struct_idx][T] = (times, n_dimers, min_oo, final_atoms)
results = {}

for struct_idx, atoms_orig in dimer_structures:
    results[struct_idx] = {}
    n_dim_init, min_oo_init, _ = count_oo_dimers(atoms_orig, cutoff=dimer_cutoff)
    print(f"\n{'='*60}")
    print(f"Structure {struct_idx + 1} — initial dimers: {n_dim_init}, min O-O: {min_oo_init:.3f} Å")
    print(f"{'='*60}")

    for T in temperatures:
        print(f"\n  T = {T} K ...")
        work = base_dir / f"eq_struct{struct_idx+1:02d}_T{T}K"
        work.mkdir(parents=True, exist_ok=True)

        # Write input structure
        input_file = work / "input.vasp"
        write(str(input_file), atoms_orig, format="vasp")

        # Run equilibration using AmorphGen's eq_run
        # eq_run signature: run(atoms, calc, stage, cfg_override, work_dir)
        atoms_eq = atoms_orig.copy()
        atoms_eq.calc = calc

        eq_atoms = eq_run(
            atoms_eq,
            calc=calc,
            stage="high",  # Uses high-T equilibration stage
            cfg_override={"eq_high": {
                "ensemble": "nvt",
                "T": T,
                "steps": eq_steps,
                "timestep": eq_timestep,
            }},
            work_dir=str(work),
        )

        # Analyse trajectory — search recursively for the trajectory file
        traj_file = None
        candidates = (
            list(work.rglob("*eq*.xyz"))
            + list(work.rglob("*eq*.extxyz"))
            + list(work.rglob("*.xyz"))
            + list(work.rglob("*.extxyz"))
        )
        # Filter out final/output files, keep trajectory files
        for c in candidates:
            if "opt" not in c.name:
                traj_file = c
                break
        if traj_file:
            print(f"    Found trajectory: {traj_file}")

        if traj_file is not None and traj_file.exists():
            times, n_dims, min_oos = track_dimers_in_trajectory(
                traj_file, cutoff=dimer_cutoff, stride=10
            )
            n_dim_final, min_oo_final, _ = count_oo_dimers(eq_atoms, cutoff=dimer_cutoff)
            results[struct_idx][T] = {
                "times": times,
                "n_dimers": n_dims,
                "min_oo": min_oos,
                "final_atoms": eq_atoms,
                "n_dim_final": n_dim_final,
                "min_oo_final": min_oo_final,
            }
            print(f"    Dimers: {n_dim_init} → {n_dim_final} | Min O-O: {min_oo_init:.3f} → {min_oo_final:.3f} Å")
        else:
            print(f"    WARNING: Trajectory file not found at {traj_file}")
            results[struct_idx][T] = None

        # Save final structure
        write(str(work / "final.vasp"), eq_atoms, format="vasp")

print("\nAll equilibrations complete.")


Structure 1 — initial dimers: 5, min O-O: 1.467 Å

  T = 4000 K ...
[Stage 4] Using provided Atoms object
[Stage 4] NVT equilibration  T=4000 K  2000 steps (4.0 ps)
       0      0.0000    3845.5     -201.5328       19.8829     -181.6499      673.01
     100      0.2000    4946.8     -191.7201       25.5770     -166.1431      673.01
     200      0.4000    5464.9     -184.7154       28.2557     -156.4596      673.01
     300      0.6000    4661.5     -185.1171       24.1020     -161.0151      673.01
     400      0.8000    4190.6     -188.6900       21.6673     -167.0227      673.01
     500      1.0000    5490.6     -188.3795       28.3887     -159.9908      673.01
     600      1.2000    3828.6     -188.9535       19.7955     -169.1580      673.01
     700      1.4000    5727.6     -183.1135       29.6138     -153.4997      673.01
     800      1.6000    4090.3     -184.2646       21.1483     -163.1164      673.01
     900      1.8000    3965.8     -185.9725       20.5048     -165.4

## 6. Results: Dimer Dissociation Kinetics

### 6.1 Dimer Count vs. Time

Track how the number of O–O dimers evolves during equilibration at each temperature.
If the Arrhenius estimate is correct, dimers should dissociate rapidly at T ≥ 2000 K
but persist at lower temperatures within our 20 ps window.

In [8]:
# Colour map for temperatures
cmap = plt.cm.plasma
temp_colors = {T: cmap(i / (len(temperatures) - 1)) for i, T in enumerate(temperatures)}

fig, axes = plt.subplots(1, len(dimer_structures), figsize=(5 * len(dimer_structures), 4),
                          squeeze=False, sharey=True)

for col, (struct_idx, _) in enumerate(dimer_structures):
    ax = axes[0, col]
    
    for T in temperatures:
        res = results[struct_idx].get(T)
        if res is None:
            continue
        ax.plot(res["times"], res["n_dimers"],
                color=temp_colors[T], label=f"{T} K", linewidth=1.5)
    
    ax.set_xlabel("Time (ps)")
    ax.set_title(f"Structure {struct_idx + 1}")
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    if col == 0:
        ax.set_ylabel("Number of O–O dimers")
    ax.legend(fontsize=8)
    ax.set_xlim(0, eq_time_ps)

fig.suptitle("O–O Dimer Count vs. Time at Different Temperatures", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(str(base_dir / "dimer_count_vs_time.png"), dpi=150, bbox_inches="tight")
plt.show()

ZeroDivisionError: division by zero

### 6.2 Minimum O–O Distance vs. Time

Track the minimum O–O distance over the trajectory. Dimer dissociation is signalled
by the minimum O–O distance rising above ~2.0 Å (from the initial ~1.47 Å peroxide bond).

In [ ]:
fig, axes = plt.subplots(1, len(dimer_structures), figsize=(5 * len(dimer_structures), 4),
                          squeeze=False, sharey=True)

for col, (struct_idx, _) in enumerate(dimer_structures):
    ax = axes[0, col]
    
    for T in temperatures:
        res = results[struct_idx].get(T)
        if res is None:
            continue
        ax.plot(res["times"], res["min_oo"],
                color=temp_colors[T], label=f"{T} K", linewidth=1.0, alpha=0.8)
    
    # Reference lines
    ax.axhline(y=dimer_cutoff, color="gray", linestyle="--", linewidth=0.8, label=f"Cutoff ({dimer_cutoff} Å)")
    ax.axhline(y=1.47, color="red", linestyle=":", linewidth=0.8, label="Peroxide (1.47 Å)")
    
    ax.set_xlabel("Time (ps)")
    ax.set_title(f"Structure {struct_idx + 1}")
    if col == 0:
        ax.set_ylabel("Min O–O distance (Å)")
    ax.legend(fontsize=7, loc="lower right")
    ax.set_xlim(0, eq_time_ps)
    ax.set_ylim(1.0, 3.5)

fig.suptitle("Minimum O–O Distance vs. Time", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(str(base_dir / "min_oo_vs_time.png"), dpi=150, bbox_inches="tight")
plt.show()

### 6.3 Final Dimer Count by Temperature

Bar chart showing how many dimers remain after 20 ps equilibration at each temperature,
averaged across all structures.

In [ ]:
# Compute average final dimer count per temperature
avg_dimers_final = []
std_dimers_final = []
avg_dimers_initial = []

for T in temperatures:
    finals = []
    initials = []
    for struct_idx, atoms_orig in dimer_structures:
        n_dim_init, _, _ = count_oo_dimers(atoms_orig, cutoff=dimer_cutoff)
        initials.append(n_dim_init)
        res = results[struct_idx].get(T)
        if res is not None:
            finals.append(res["n_dim_final"])
        else:
            finals.append(n_dim_init)  # No data — assume unchanged
    avg_dimers_final.append(np.mean(finals))
    std_dimers_final.append(np.std(finals))
    avg_dimers_initial.append(np.mean(initials))

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(temperatures))
width = 0.35

bars_init = ax.bar(x - width/2, avg_dimers_initial, width, label="Before equilibration",
                    color="salmon", edgecolor="darkred", alpha=0.8)
bars_final = ax.bar(x + width/2, avg_dimers_final, width, yerr=std_dimers_final,
                     label="After 4 ps equilibration", color="steelblue",
                     edgecolor="navy", alpha=0.8, capsize=3)

ax.set_xlabel("Temperature (K)")
ax.set_ylabel("Mean number of O–O dimers")
ax.set_title("Dimer Dissociation: Effect of Temperature")
ax.set_xticks(x)
ax.set_xticklabels([str(T) for T in temperatures])
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend()
plt.tight_layout()
plt.savefig(str(base_dir / "dimer_bar_chart.png"), dpi=150, bbox_inches="tight")
plt.show()

### 6.4 Arrhenius Analysis

Estimate the dissociation timescale from the MD trajectories and compare with the
Arrhenius prediction ($E_a \approx 1$ eV, $\nu \approx 10^{13}$ Hz).

For each temperature, we estimate $\tau$ as the time at which the dimer count first
drops to zero (or, if it never reaches zero, report $\tau > t_{\mathrm{sim}}$).

In [ ]:
kB = 8.617333e-5  # eV/K
nu = 1e13          # Hz (attempt frequency)
Ea_guess = 1.0     # eV (assumed barrier)

# Arrhenius prediction
T_arr = np.linspace(2000, 5000, 200)
tau_arr = (1.0 / nu) * np.exp(Ea_guess / (kB * T_arr))  # seconds
tau_arr_ps = tau_arr * 1e12  # convert to ps

# Extract dissociation times from MD
tau_md = {T: [] for T in temperatures}

for struct_idx, _ in dimer_structures:
    for T in temperatures:
        res = results[struct_idx].get(T)
        if res is None:
            tau_md[T].append(np.inf)
            continue
        
        # Find first frame where dimers = 0
        zero_idx = np.where(res["n_dimers"] == 0)[0]
        if len(zero_idx) > 0:
            tau_md[T].append(res["times"][zero_idx[0]])
        else:
            tau_md[T].append(np.inf)  # Dimers never fully dissociated

# Average tau per temperature (only finite values)
tau_md_avg = []
tau_md_finite = []
for T in temperatures:
    taus = [t for t in tau_md[T] if np.isfinite(t)]
    if taus:
        tau_md_avg.append(np.mean(taus))
        tau_md_finite.append(True)
    else:
        tau_md_avg.append(eq_time_ps)  # Lower bound
        tau_md_finite.append(False)

# Plot
fig, ax = plt.subplots(figsize=(7, 5))

# Arrhenius prediction line
ax.semilogy(1000 / T_arr, tau_arr_ps, "k--", linewidth=1.5,
            label=f"Arrhenius ($E_a$ = {Ea_guess} eV)")

# MD data points
for i, T in enumerate(temperatures):
    marker = "o" if tau_md_finite[i] else "^"
    color = temp_colors[T]
    label_str = f"{T} K"
    if not tau_md_finite[i]:
        label_str += " (> sim time)"
    ax.semilogy(1000 / T, tau_md_avg[i], marker, color=color,
                markersize=10, markeredgecolor="black", markeredgewidth=0.8,
                label=label_str, zorder=5)

# MD time window
ax.axhline(y=eq_time_ps, color="gray", linestyle=":", linewidth=0.8, alpha=0.7)
ax.text(0.25, eq_time_ps * 1.3, f"Simulation limit ({eq_time_ps} ps)",
        fontsize=8, color="gray")

ax.set_xlabel("1000 / T (K⁻¹)")
ax.set_ylabel("Dissociation timescale τ (ps)")
ax.set_title("Arrhenius Analysis of O–O Dimer Dissociation")
ax.legend(fontsize=8, loc="upper left")
ax.set_xlim(0.2, 0.55)
ax.set_ylim(0.1, 1e3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(base_dir / "arrhenius_plot.png"), dpi=150, bbox_inches="tight")
plt.show()

### 6.5 Partial RDF: Before vs. After Equilibration

Compare O–O partial RDF before and after equilibration at the highest temperature.
Dimer dissociation should be visible as the disappearance of the peak near 1.47 Å.

In [ ]:
from ase.geometry.analysis import Analysis

def compute_partial_rdf(atoms, elem1, elem2, rmax=4.0, nbins=100):
    """Compute partial RDF between two element types."""
    ana = Analysis(atoms)
    
    idx1 = [i for i, s in enumerate(atoms.get_chemical_symbols()) if s == elem1]
    idx2 = [i for i, s in enumerate(atoms.get_chemical_symbols()) if s == elem2]
    
    # Use get_distances for PBC-correct distances
    p1 = atoms.positions[idx1]
    p2 = atoms.positions[idx2]
    _, D = get_distances(p1, p2, cell=atoms.cell, pbc=True)
    
    dists = []
    for ii in range(len(idx1)):
        for jj in range(len(idx2)):
            if elem1 == elem2 and idx1[ii] >= idx2[jj]:
                continue  # Avoid double-counting for same element
            d = np.linalg.norm(D[ii, jj])
            if 0 < d < rmax:
                dists.append(d)
    
    # Histogram and normalise to RDF
    bins = np.linspace(0, rmax, nbins + 1)
    hist, edges = np.histogram(dists, bins=bins)
    r = 0.5 * (edges[:-1] + edges[1:])
    dr = edges[1] - edges[0]
    
    # Normalise: RDF shell volume
    vol = atoms.get_volume()
    n1 = len(idx1)
    n2 = len(idx2) if elem1 != elem2 else len(idx2) - 1
    rho = n2 / vol
    shell_vol = 4.0 * np.pi * r**2 * dr
    g_r = hist / (n1 * rho * shell_vol + 1e-30)
    
    return r, g_r


# Pick the highest temperature and first dimer structure
T_highest = max(temperatures)
struct_idx_plot, atoms_before = dimer_structures[0]
res_highest = results[struct_idx_plot].get(T_highest)

pairs = [("O", "O"), ("In", "O"), ("In", "In")]
pair_labels = ["O–O", "In–O", "In–In"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for j, (e1, e2) in enumerate(pairs):
    ax = axes[j]
    
    r_before, g_before = compute_partial_rdf(atoms_before, e1, e2)
    ax.plot(r_before, g_before, "r-", linewidth=1.5, label="Before (optimised)")
    
    if res_highest is not None:
        atoms_after = res_highest["final_atoms"]
        r_after, g_after = compute_partial_rdf(atoms_after, e1, e2)
        ax.plot(r_after, g_after, "b-", linewidth=1.5, label=f"After {T_highest} K eq.")
    
    if j == 0:  # O-O panel: mark dimer region
        ax.axvspan(1.2, dimer_cutoff, alpha=0.1, color="red", label="Dimer region")
    
    ax.set_xlabel("r (Å)")
    ax.set_ylabel("g(r)")
    ax.set_title(pair_labels[j])
    ax.legend(fontsize=8)
    ax.set_xlim(0, 4.0)

fig.suptitle(f"Partial RDF — Structure {struct_idx_plot+1}, Before vs. After {T_highest} K Equilibration",
             fontsize=12, y=1.03)
plt.tight_layout()
plt.savefig(str(base_dir / "rdf_before_after.png"), dpi=150, bbox_inches="tight")
plt.show()

### 6.6 O–O Distance Distribution After Equilibration

Histogram of all O–O distances < 4 Å for the final snapshot at each temperature.
Shows how the short-range O–O structure evolves with temperature.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

# Before equilibration (first dimer structure)
struct_idx_plot, atoms_before = dimer_structures[0]
o_idx = [i for i, s in enumerate(atoms_before.get_chemical_symbols()) if s == "O"]
p_o = atoms_before.positions[o_idx]
_, D_before = get_distances(p_o, cell=atoms_before.cell, pbc=True)
n_o = len(o_idx)
dists_before = [np.linalg.norm(D_before[ii, jj])
                for ii in range(n_o) for jj in range(ii+1, n_o)
                if 0 < np.linalg.norm(D_before[ii, jj]) < 4.0]

ax.hist(dists_before, bins=50, range=(1.0, 4.0), alpha=0.4, color="gray",
        density=True, label="Before eq.", edgecolor="gray")

# After equilibration at each temperature
for T in temperatures:
    res = results[struct_idx_plot].get(T)
    if res is None:
        continue
    atoms_T = res["final_atoms"]
    o_idx_T = [i for i, s in enumerate(atoms_T.get_chemical_symbols()) if s == "O"]
    p_o_T = atoms_T.positions[o_idx_T]
    _, D_T = get_distances(p_o_T, cell=atoms_T.cell, pbc=True)
    n_o_T = len(o_idx_T)
    dists_T = [np.linalg.norm(D_T[ii, jj])
               for ii in range(n_o_T) for jj in range(ii+1, n_o_T)
               if 0 < np.linalg.norm(D_T[ii, jj]) < 4.0]
    
    ax.hist(dists_T, bins=50, range=(1.0, 4.0), alpha=0.5,
            color=temp_colors[T], density=True, label=f"{T} K",
            histtype="step", linewidth=1.5)

ax.axvline(x=1.47, color="red", linestyle=":", linewidth=0.8, label="O₂²⁻ bond")
ax.axvline(x=dimer_cutoff, color="gray", linestyle="--", linewidth=0.8, label=f"Cutoff ({dimer_cutoff} Å)")

ax.set_xlabel("O–O distance (Å)")
ax.set_ylabel("Density")
ax.set_title(f"O–O Distance Distribution — Structure {struct_idx_plot+1}")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(str(base_dir / "oo_dist_histograms.png"), dpi=150, bbox_inches="tight")
plt.show()

## 7. Summary Table

In [ ]:
print(f"{'Temperature (K)':>16s}  {'Init dimers':>12s}  {'Final dimers':>13s}  "
      f"{'Min O-O init':>13s}  {'Min O-O final':>14s}  {'τ_diss (ps)':>12s}")
print("-" * 90)

for T in temperatures:
    init_dims = []
    final_dims = []
    init_oos = []
    final_oos = []
    taus = []
    
    for struct_idx, atoms_orig in dimer_structures:
        n_init, min_init, _ = count_oo_dimers(atoms_orig, cutoff=dimer_cutoff)
        init_dims.append(n_init)
        init_oos.append(min_init)
        
        res = results[struct_idx].get(T)
        if res is not None:
            final_dims.append(res["n_dim_final"])
            final_oos.append(res["min_oo_final"])
            # Dissociation time
            zero_idx = np.where(res["n_dimers"] == 0)[0]
            if len(zero_idx) > 0:
                taus.append(res["times"][zero_idx[0]])
            else:
                taus.append(np.inf)
        else:
            final_dims.append(n_init)
            final_oos.append(min_init)
            taus.append(np.inf)
    
    mean_init = np.mean(init_dims)
    mean_final = np.mean(final_dims)
    mean_oo_init = np.mean(init_oos)
    mean_oo_final = np.mean(final_oos)
    finite_taus = [t for t in taus if np.isfinite(t)]
    tau_str = f"{np.mean(finite_taus):.1f}" if finite_taus else f"> {eq_time_ps}"
    
    print(f"{T:>16d}  {mean_init:>12.1f}  {mean_final:>13.1f}  "
          f"{mean_oo_init:>13.3f}  {mean_oo_final:>14.3f}  {tau_str:>12s}")

## 8. Discussion

### Key observations

**Temperature dependence of dimer dissociation:**
- At 3000–4000 K, the Arrhenius estimate predicts $\tau \approx$ 2–5 ps, which is
  within our 4 ps simulation window.
- Comparing 3000, 3500, and 4000 K reveals whether dimers break faster at higher T,
  as expected from the exponential temperature dependence.
- Any dimers surviving at 3000 K but not at 4000 K would bracket the effective barrier.

### Implications for melt-and-quench workflows

1. **Melting temperature matters:** The standard melt temperature of 2000–3000 K in
   AmorphGen's pipeline is sufficient to dissociate O–O dimers introduced by random
   placement. This validates the hybrid workflow (random gen → high-T equilibration).

2. **Quench rate effects:** If dimers form during quenching (unlikely but possible
   at very high cooling rates), they will persist at low temperatures. The barrier
   height determines whether DFT relaxation can remove them.

3. **MLIP choice matters:** MACE supports dimers while CHGNet does not. For studies
   of oxygen defect chemistry, the choice of MLIP fundamentally affects the accessible
   physics. DFT validation is recommended.

### Limitations

- Small system size (40 atoms) — limited statistics and finite-size effects on RDF.
- Only 5 structures per system — increase to 20–50 for publication quality.
- Barrier height ($E_a \approx 1$ eV) is assumed — would need NEB calculations to confirm.
- NVT ensemble fixes volume — NPT would allow density relaxation alongside dimer breaking.

### Next steps

- **DFT single-point validation:** Compare MACE and CHGNet energetics for dimer vs.
  non-dimer structures at the DFT level.
- **NEB barrier calculation:** Determine the actual $E_a$ for O–O dimer dissociation
  in amorphous In₂O₃ using MACE or DFT.
- **Larger system sizes:** Repeat with 80–160 atom cells for better statistics.
- **Production runs:** Use longer equilibration times (50–100 ps) at 2000 K to
  pin down the exact dissociation timescale.

## 9. CLI Equivalent Commands

The workflow above can be partially reproduced using AmorphGen's CLI:

```bash
# Step 1: Generate random structures with loose O-O minsep
amorphgen --random-gen \
    --composition In=16,O=24 \
    --target-density 5.5 \
    --minsep In-In=2.8,In-O=1.9,O-O=1.5 \
    --n-structures 5 \
    --no-relax \
    --max-attempts 50000 \
    --format vasp \
    --work-dir random_loose

# Step 2: Optimise with MACE (cubic cell filter)
amorphgen random_loose/random_001.vasp \
    --stages 1 \
    --model mace-mpa-0-medium \
    --default-dtype float64 \
    --device cpu \
    --cell-filter cubic \
    --fmax 0.05 \
    --opt-steps 500 \
    --work-dir opt_MACE

# Step 3: Equilibrate at high temperature
# (run for each temperature — currently needs separate calls)
amorphgen opt_MACE/stage1_opt/opt_final.vasp \
    --stages 4 \
    --model mace-mpa-0-medium \
    --default-dtype float64 \
    --device cpu \
    --eq-high-ensemble nvt \
    --eq-high-T 3000 \
    --eq-high-steps 2000 \
    --work-dir eq_3000K
```

**Note:** The Python API (used in this notebook) provides more flexibility for
looping over temperatures and tracking dimer counts from trajectories.

## 10. Save All Figures and Structures

In [ ]:
# Save final structures at highest temperature
T_save = max(temperatures)
for struct_idx, _ in dimer_structures:
    res = results[struct_idx].get(T_save)
    if res is not None:
        for fmt, ext in [("vasp", ".vasp"), ("cif", ".cif"), ("extxyz", ".xyz")]:
            out = base_dir / f"struct{struct_idx+1:02d}_after_{T_save}K{ext}"
            write(str(out), res["final_atoms"], format=fmt)

print(f"Saved final structures at {T_save} K in {base_dir}/")
print(f"\nAll figures saved in {base_dir}/:")
for f in sorted(base_dir.glob("*.png")):
    print(f"  {f.name}")